# Single-qubit DB on a transmon — summary

A trimmed version of [`1qb-DB-transmon.ipynb`](1qb-DB-transmon.ipynb) for a reader who
already knows transmon calibration and deterministic benchmarking, and wants to see
what geometric (space-curve) pulse design buys on a real device. Every number and
figure here is a subset of that notebook: nothing is re-optimized, no new model is
introduced, and the numerical implementation is unchanged
([`db_transmon.py`](db_transmon.py), [`db_helpers.py`](db_helpers.py)).

Three waveforms, all strictly resonant $X_\pi$ gates:

- **Gaussian** — the baseline;
- **resonant composite** — first-order robust against a multiplicative drive error by
  construction, and optimized on a finite $\pm3\%$ error window;
- **BARQ resonant robust** — the space-curve optimizer's own first-order robust
  resonant solution, produced for this device.

In [ ]:
from pathlib import Path
import sys

search_roots = (Path.cwd(), *Path.cwd().parents)
DEMO_DIR = next(
    path
    for root in search_roots
    for path in (root, root / "proposals/1qb-DB/1qb-DB-demo")
    if (path / "db_transmon.py").is_file()
)
sys.path.insert(0, str(DEMO_DIR))

import db_helpers as db
import db_transmon as tr

# The first two come from the shared waveform store; the BARQ pulse was optimized
# for this device in section 1 of the full notebook.
pulses = {
    **tr.load_imported_waveforms(["Gaussian", "Resonant composite"]),
    "BARQ resonant robust": tr.load_transmon_waveform(
        "barq_resonant_robust_a0.1"
    ),
}
COLORS = tr.color_cycle(pulses)
print(tr.DEVICE)

## 1. The waveforms

One gauge has to be fixed before waveforms of different shape can be compared. A
normalized waveform $(t/T_g,\;T_g\Omega)$ does not say how hard to drive: rescaling
the underlying space curve trades $T_g$ against $\Omega$ and leaves every
dimensionless quantity unchanged. We fix it the way a laboratory does — **every pulse
is driven at the same peak Rabi rate**, $\Omega_{\max}/2\pi = 50$ MHz — so the price
of robustness appears as gate time, and leakage stays comparable across the
comparison.

In [ ]:
figure = tr.plot_physical_controls(pulses, colors=COLORS)

## 2. What the geometry is, and what it costs

The injected error is multiplicative,
$(\Omega_x,\Omega_y)\to(1+\epsilon)(\Omega_x,\Omega_y)$, so it reaches the qubit
through the drive Hamiltonian written in the toggling frame of the ideal gate and
integrated — the **error curve**

$$R(t)=\int_0^t\vec g(s)\,\mathrm{d}s,
\qquad U_0^\dagger(s)\,H_c(s)\,U_0(s)=\tfrac12\,\vec g(s)\cdot\vec\sigma .$$

To first order in $\epsilon$ the entire error of the gate is the *endpoint* of that
curve; to second order it is the *area* it encloses. Pulse design becomes curve
drawing, which is the whole reason for the geometric language.

| quantity | formula | what it means |
|---|---|---|
| gate | curve length $\int_0^{T_g}\Omega\,\mathrm{d}t$ | total rotation; $\pi$ for a bare $X_\pi$, necessarily more for a robust one |
| control error, 1st order | $\lvert R(T_g)-R(0)\rvert$ — the *closure* | the error rotation angle per unit $\epsilon$; zero means the curve closes, i.e. first-order robust |
| control error, 2nd order | $\lvert\vec A\rvert$, $\vec A=\tfrac12\oint R\times\mathrm{d}R$ | what is left once the curve is closed |
| dephasing, 1st order | closure of the same construction with $G=Z/2$ | robustness against a static frequency offset $\delta_z$ |
| drive cost | $T_g\Omega_{\max}$ | gauge invariant; with the peak drive fixed it *is* the gate time |

Every column except $T_g$ and $P_2$ is dimensionless and scale invariant.

In [ ]:
curves = db.calculate_error_curves(pulses)
figure = db.plot_error_curves(curves, colors=COLORS)

The Gaussian's curve is a straight line of length $\pi$: it encloses no area, but it
fails at first order by the full $\pi$ — zero area is not robustness. The composite
almost closes. The BARQ curve closes too, but leaves the plane, and the area it
encloses is large.

In [ ]:
metrics = tr.print_geometry_table(pulses)

Read the table as a set of corners rather than a ranking: the composite owns the
control channel at both orders, the BARQ pulse owns the dephasing channel (closure
$5\times10^{-5}$, which the closed space curve gives for free) and pays for it in
leakage, and the Gaussian owns leakage and nothing else. First-order control
robustness costs a total rotation of $5\pi$ against the Gaussian's $\pi$, hence
$100$ ns against $22$ ns at the same peak drive.

## 3. Calibration

The waveforms are exact $X(\pi)$ in the two-level model, and on three levels they are
not: at $\Omega_{\max}/\lvert\alpha\rvert=0.25$ the $\lvert2\rangle$ level
Stark-shifts the 0–1 transition while the drive is on, which both mis-scales the
rotation angle and tilts the rotation axis by more than the error we intend to
inject. Three knobs are fitted once per waveform, at $\epsilon=0$, and then held
fixed: **drive amplitude**, **drive frequency** and a **virtual $Z$**, bounded to
$\pm25\%$ and $\pm30$ MHz. The objective is the rotation error inside the qubit
subspace with the leakage loss divided out, so the fit cannot buy fidelity by hiding
in one of the narrow leakage nulls. No DRAG and no reshaping; the injected $\epsilon$
is applied on top of the calibrated amplitude, which is what an unknown residual
control error means.

In [ ]:
calibrations = tr.print_calibration(pulses)

The frequency correction lands at $4$–$6$ MHz, the order of the Stark shift
$\Omega^2/4\lvert\alpha\rvert$, and two or three knobs remove the rotation error
almost completely. What remains in `1-F` is leakage, which calibration cannot
touch.

## 4. Deterministic benchmarking

One cycle is the pair $XX$ — ideally the identity — so the readout is the return
probability $P_0(n)$, and $n=60$ cycles is $120$ gates. The error model is frozen,
one value each and never swept: $T_1=T_2^{\rm echo}=60\ \mu$s, a static
$\delta_z/2\pi = 0.1$ MHz, and an unknown residual control error $\epsilon=\pm3\%$.
One gate is integrated once into a $9\times9$ channel, so the sequence is a matrix
power of it.

Because decoherence pulls $P_0$ down whether or not the control is robust, the run is
repeated with the incoherent terms switched off, and the $\epsilon=0$ traces are kept
as baselines. That gives two independent readings instead of one $\Delta P_0$: the
**oscillation** (control error alone) and the **envelope loss** (what the gate
duration costs).

In [ ]:
traces = tr.db_traces(pulses, calibrations=calibrations)
figure = tr.plot_db_traces(traces, colors=COLORS, layout="tall")

Panel (a) is the standard readout; the dashed lines are the $\epsilon=0$ envelopes.
The signal is the **period**, not the amplitude: run long enough and every waveform
sweeps the full range of $P_0$. The Gaussian's period is $34$ cycles — exactly the
$1/\epsilon$ that $\theta_g=\epsilon\pi$ per gate predicts — the BARQ pulse's is
$38$, and the composite's is $174$, so panel (a) shows only its first third.

Panel (b) is the same traces against physical time, which is the control for the
gate-time difference: per microsecond the three envelopes decay at
$8$–$9\times10^{-3}$, i.e. at the rate $T_1$ and $T_2$ set, independent of the
waveform. Panel (c) is accumulated leakage. Panel (d) divides the envelope out and
puts the remaining control-error signal on a log-log axis, where it grows as $n^2$
(coherent errors add in amplitude) and the ranking is the horizontal offset between
parallel lines.

In [ ]:
readouts = tr.print_db_table(traces)

## 5. What to take away

1. **The geometry works, and the size of "works" is the result.** The composite needs
   $13$ cycles to build a $5\%$ signal where the Gaussian needs $3$, and its
   oscillation period is $5.1\times$ longer. Per gate its error angle is
   $0.035$ rad against the Gaussian's $0.093$ rad — and $0.093$ rad is simply
   $3\%$ of $180^\circ$, the whole of an unprotected gate's error.

2. **But a factor $4$–$5$ is not the factor $10^5$ the design promises, and the gap
   is a channel the error curve does not describe.** The injected $\epsilon$ also
   rescales the Stark shift $\Omega^2/4\lvert\alpha\rvert$, and the drive-frequency
   knob cancels that only at $\epsilon=0$; the residual is therefore first order in
   $\epsilon$ however well the curve closes. The closure predicts
   $1.4\times10^{-4}$ rad per gate for the composite and the sequence measures
   $3.5\times10^{-2}$. Switching the third level off restores the two-level answer
   ($\Delta P_0 = 3.9\times10^{-6}$ over the same $60$ cycles), which is how we know
   that this is the mechanism and not a numerical artefact.

3. **Robustness is paid for in gate time, and the exchange rate is geometric.**
   $100$ ns against $22$ ns at the same peak drive, so per gate the composite pays
   $4.5\times$ the decoherence: envelope loss $9.8\times10^{-2}$ against
   $2.4\times10^{-2}$ over $60$ cycles. Per unit *time* the two are identical, so
   this is a cost per gate, not a worse device.

4. **The BARQ pulse gains nothing here, for two reasons that are both visible above.**
   Its control-error curve closes ($5.6\times10^{-3}$), but its second-order area is
   $3.9$ and its leakage floor is $1.5\times10^{-2}$ — an order of magnitude above the
   composite's total error before any noise is switched on. First-order robustness in
   the wrong channel, with a $\lvert2\rangle$ problem on top.

**The lever this points at.** First-order robustness was imposed in the *two-level*
toggling frame. The term that actually limits these pulses is the
$\epsilon$-dependent $Z$ generator the third level induces, so the natural next design
target is a three-level toggling-frame error curve — or the gauge itself: lowering
$\Omega_{\max}/\lvert\alpha\rvert$ weakens the Stark term quadratically while
decoherence grows only linearly in $T_g$.